# Getting started with Braidworks

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rafaelmdc/braidworks/blob/main/notebooks/getting-started.ipynb)

Braidworks looks up fields for a list of identifiers across biological databases. You
say what your IDs are and which fields you want, and it finds a route across the
sources and returns one row per ID.

In this notebook you will:

1. install Braidworks;
2. find the fields you need by describing them;
3. look up two genes (and one non-gene) across UniProt, PDBe and Reactome;
4. resolve a match the source was unsure about;
5. export the table and cite the sources.

It needs an internet connection; the sources used are free public APIs and need no
account. Running every cell takes a minute or two.

## 1. Install

One package installs the planner, every public source and the browser interface. It
downloads no biological databases.

In [ ]:
%pip install -q "braidworks[pandas]"

In [ ]:
import braidworks as bw

## 2. Find the fields you want

Every value has a typed field name, such as `protein.name`. You do not need to know
them in advance: describe what you want in plain words.

In [ ]:
bw.find("pdb structures")

In [ ]:
bw.find("pathway names")

`structure.pdb.ids` lists a protein's experimental structures and
`pathway.reactome.names` the Reactome pathways it takes part in. Your IDs also need a
type: gene symbols, protein names and accessions are all `protein.query`.

## 3. Look up your IDs

`organism="9606"` (the NCBI taxon ID for human) makes the symbols mean the human genes.
`NOTAGENE` is there on purpose, so you can see how a miss is reported.

In [ ]:
result = bw.get(
    ["TP53", "BRCA1", "NOTAGENE"],
    have="protein.query",
    want=["protein.name", "structure.pdb.ids", "pathway.reactome.names"],
    organism="9606",
)
print(result.summary())
result.to_pandas()

Every input gets a row, with a `status`: `resolved`, `unresolved` (the source had no
match: NOTAGENE), `review` (a person must choose) or `error` (a source failed).
`missing` names the fields a row lacks.

A list field comes back whole. It behaves as a Python list and also knows its true
size, which `to_pandas()` and `to_csv()` keep in a `.total` column:

In [ ]:
structures = result.rows[0]["structure.pdb.ids"]
print(len(structures), "structures for TP53, for example", list(structures[:5]))

`result.route` shows which sources answered, and in what order:

In [ ]:
for braid in result.route:
    print(braid)

## 4. Resolve an uncertain match

When a source cannot tell which entity you meant, Braidworks pauses that row instead
of guessing. Here a misspelled species name matches several taxa:

In [ ]:
taxa = bw.get(
    ["Bacillus", "Escherichia coli"],
    have="organism.name",
    want=["ncbi.taxon.id", "ncbi.taxon.rank"],
)
print(taxa.summary())
taxa.review[0]

Read the candidates, record a decision, then resume. Only the decided rows run again;
the rows that had already finished are kept. Choose by what the candidate is, not by
its position: the order comes from the source.

In [ ]:
item = taxa.review[0]
bacteria = [c["index"] for c in item.candidates if str(c["values"]["ncbi.taxon.id"]) == "1386"]
if bacteria:
    item.choose(bacteria[0])
else:  # the source matched one taxon but flagged it: confirm it or reject it
    item.accept()
taxa = taxa.resume()
print(taxa.summary())
taxa.to_pandas()

## 5. Export and cite

The CSV is the same table the command line (`braidworks get --format csv`) and the
browser download give for the same query.

In [ ]:
result.to_csv("braidworks-results.csv")
result.to_pandas().head()

In [ ]:
try:  # on Colab, download the file to your computer
    from google.colab import files

    files.download("braidworks-results.csv")
except ImportError:
    print("Saved braidworks-results.csv")

Cite the sources your results drew on:

In [ ]:
print(result.cite())

## Next steps

- No connection? `bw.demo("proteins")` runs a canned query on samples bundled with the
  sources, with no network.
- [Your first analysis in Python](https://github.com/rafaelmdc/braidworks/blob/main/docs/tutorials/first-analysis-in-python.md)
  goes further with sessions, caching and batches.
- [Work in Python](https://github.com/rafaelmdc/braidworks/blob/main/docs/how-to/work-in-python.md)
  collects recipes, and the [Python API reference](https://github.com/rafaelmdc/braidworks/blob/main/docs/reference/python-api.md)
  lists every function.